# MD sandbox — turn the knobs

A small, fast MD playground: swap the protein, temperature, seed, and (in Section 2) the force field, water model, ensemble, thermostat, and timestep — and watch how the physics responds. All runs are **in memory** (no files). The prep + run machinery lives in `mdtsandbox.py` (which reuses `mdtutorial`), so this notebook stays about the *choices*, not the plumbing.

In [ ]:
# --- environment on-ramp: make sure the MD stack + the modules are importable in THIS kernel ---
import importlib.util, sys, os, subprocess
_missing = [m for m in ("openmm", "pdbfixer", "mdtraj", "py3Dmol") if importlib.util.find_spec(m) is None]
if _missing and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj", "py3Dmol"], check=False)
    _missing = [m for m in _missing if importlib.util.find_spec(m) is None]
if _missing:
    raise SystemExit(f"Missing in this kernel: {_missing}. Select your MD-tutorial conda kernel "
                     "(Kernel > Change Kernel). If it isn't built yet: conda env create -f environment.yml; "
                     "if it exists but is stale: conda env update -f environment.yml.")
_BASE = os.environ.get("MDTUTORIAL_BASE", "https://raw.githubusercontent.com/OWNER/REPO/main")
for _mod in ("mdtutorial.py", "mdtsandbox.py"):          # grab the shipped modules if they aren't alongside
    if not os.path.exists(_mod) and importlib.util.find_spec(_mod[:-3]) is None:
        import urllib.request
        try: urllib.request.urlretrieve(f"{_BASE}/{_mod}", _mod); print("fetched", _mod)
        except Exception as e: print("Place", _mod, "next to this notebook.", e)
import numpy as np, matplotlib.pyplot as plt
import mdtsandbox as sb

In [ ]:
# the tested PDB menu (any RCSB id also works off-menu)
print("available PDBs:")
for k, v in sb.PDB_MENU.items():
    print(f"  {k}  {v}")

## Section 1 — the basic knobs

In [ ]:
# SANDBOX KNOBS (edit me!)
PDB       = "1L2Y"     # a key from sb.PDB_MENU (or any RCSB id)
TEMP_K    = 300        # temperature in kelvin (try 280 vs 350)
SEED      = 2024       # same seed + same GPU model -> identical run; change it for a new trajectory
PROD_PS   = 50         # production length per run, ps (longer = more sampling, slower)
N_REPEATS = 3          # independent runs (different seeds) to see the spread
print(f"config: {PDB} @ {TEMP_K} K, {PROD_PS} ps x {N_REPEATS} repeat(s), base seed {SEED}")
print("       ", sb.PDB_MENU.get(PDB, "(off-menu — fetched from RCSB; some structures need extra cleanup)"))

In [ ]:
# run the repeats and overlay the four generic observables
runs = []; traj_last = None
for r in range(N_REPEATS):
    traj_last, cv = sb.run(PDB, TEMP_K, SEED + r, PROD_PS)
    runs.append(cv)
    print(f"repeat {r} (seed {SEED+r}): RMSD end {cv['rmsd'][-1]:.1f} A | Rg {cv['rg'].mean():.1f} A | helix {cv['helix'].mean():.2f}")

fig, ax = plt.subplots(2, 2, figsize=(11, 6.5), facecolor="white")
panels = [("rmsd", "backbone RMSD (Å)"), ("rg", "radius of gyration (Å)"),
          ("helix", "helix fraction"), ("ete", "end-to-end Cα (Å)")]
colors = ["#1E90FF", "#FF8C00", "#CC79A7", "#009E73"]
for a, (key, lab) in zip(ax.flat, panels):
    for r, cv in enumerate(runs):
        a.plot(cv["ps"], cv[key], lw=1, color=colors[r % len(colors)], label=f"repeat {r}")
    a.set_xlabel("time (ps)"); a.set_ylabel(lab)
    if key == "helix": a.set_ylim(0, 1)
ax[0, 0].legend(fontsize=8)
fig.suptitle(f"{PDB} @ {TEMP_K} K — {PROD_PS} ps × {N_REPEATS} — generic observables", fontsize=13)
fig.tight_layout(); plt.show()

In [ ]:
# watch the last trajectory (drag to rotate; press play)
import py3Dmol, tempfile, os
_p = tempfile.mktemp(suffix=".pdb"); traj_last.save_pdb(_p)
view = py3Dmol.view(width=520, height=420)
view.addModelsAsFrames(open(_p).read(), "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.animate({"loop": "forward", "interval": 80}); view.zoomTo()
os.remove(_p); view.show()

### Temperature sweep
Same structure, several temperatures, one observable overlaid.

In [ ]:
TEMPS    = [280, 300, 320, 350, 370, 390]   # kelvin
OBS      = "rmsd"                            # "rmsd" | "rg" | "helix" | "ete"
SWEEP_PS = 500                              # ps per temperature
obs_label = {"rmsd": "backbone RMSD (Å)", "rg": "radius of gyration (Å)",
             "helix": "helix fraction", "ete": "end-to-end Cα (Å)"}[OBS]
plt.figure(figsize=(7.5, 4.2), facecolor="white")
for T, col in zip(TEMPS, plt.cm.turbo(np.linspace(0.1, 0.9, len(TEMPS)))):
    _, cv = sb.run(PDB, T, SEED, SWEEP_PS)
    plt.plot(cv["ps"], cv[OBS], lw=1.4, color=col, label=f"{T} K")
    print(f"{T} K: <{OBS}> = {cv[OBS].mean():.2f}")
plt.xlabel("time (ps)"); plt.ylabel(obs_label)
plt.title(f"{PDB}: {obs_label} vs temperature"); plt.legend(title="T"); plt.tight_layout(); plt.show()

## Section 2 — change the physics
Force field + water come as matched presets; ensemble, thermostat, and timestep are separate knobs. NPT reports density; NVE reports total energy.

In [ ]:
# PHYSICS KNOBS (edit me!)
print("force-field + water presets:")
for _k in sb.FF_MENU: print("   ", _k)
FORCEFIELD  = "CHARMM36 + TIP3P"   # a key from sb.FF_MENU
ENSEMBLE    = "NPT"                # "NVE" | "NVT" | "NPT" (box breathes)
THERMOSTAT  = "Langevin"          # "Langevin" | "Nose-Hoover"  (NVT/NPT only)
TIMESTEP_FS = 2                    # 1 or 2 fs (with HBonds constraints)
PROD_PS2    = 60                   # NPT wants a little time for the box to settle
_th = "" if ENSEMBLE == "NVE" else f" ({THERMOSTAT})"
print(f"\nphysics: {PDB} @ {TEMP_K} K | {FORCEFIELD} | {ENSEMBLE}{_th} | {TIMESTEP_FS} fs | {PROD_PS2} ps")

In [ ]:
# run with the chosen physics; plot the four observables (+ the ensemble-specific one)
t2, cv2 = sb.run(PDB, TEMP_K, SEED, PROD_PS2, ff_preset=FORCEFIELD, ensemble=ENSEMBLE,
                 thermostat=THERMOSTAT, timestep_fs=TIMESTEP_FS)
extra = {"NPT": ("density", "density (g/mL)"),
         "NVE": ("total_energy", "total energy (kJ/mol)")}.get(ENSEMBLE)
panels = [("rmsd", "backbone RMSD (Å)"), ("rg", "radius of gyration (Å)"),
          ("helix", "helix fraction"), ("ete", "end-to-end Cα (Å)")]
if extra: panels[3] = extra
fig, ax = plt.subplots(2, 2, figsize=(11, 6.5), facecolor="white")
for a, (key, lab) in zip(ax.flat, panels):
    a.plot(cv2["ps"], cv2[key], lw=1.2, color="#1E90FF")
    a.set_xlabel("time (ps)"); a.set_ylabel(lab)
    if key == "helix": a.set_ylim(0, 1)
note = {"NPT": "box breathes -> density settles toward ~1 g/mL",
        "NVE": "no thermostat -> total energy should stay ~flat (integrator conservation)"}.get(ENSEMBLE, "")
_th = "" if ENSEMBLE == "NVE" else f" · {THERMOSTAT}"
fig.suptitle(f"{PDB} · {FORCEFIELD} · {ENSEMBLE}{_th} · {TIMESTEP_FS} fs" + (f"    ({note})" if note else ""), fontsize=12)
fig.tight_layout(); plt.show()

### Force-field comparison
Same protein, seed, and ensemble (NVT); two force fields.

In [ ]:
COMPARE = ["amber14 + TIP3P", "CHARMM36 + TIP3P"]   # any two keys from sb.FF_MENU
OBS2    = "rmsd"
_lab = {"rmsd": "backbone RMSD (Å)", "rg": "radius of gyration (Å)",
        "helix": "helix fraction", "ete": "end-to-end Cα (Å)"}[OBS2]
plt.figure(figsize=(7.5, 4.2), facecolor="white")
for _ffp, _col in zip(COMPARE, ["#1E90FF", "#FF8C00"]):
    _, _cvc = sb.run(PDB, TEMP_K, SEED, PROD_PS2, ff_preset=_ffp)   # NVT default
    plt.plot(_cvc["ps"], _cvc[OBS2], lw=1.4, color=_col, label=_ffp)
    print(f"{_ffp:24s}: <{OBS2}> = {_cvc[OBS2].mean():.2f}")
plt.xlabel("time (ps)"); plt.ylabel(_lab)
plt.title(f"{PDB}: {OBS2} under two force fields"); plt.legend(); plt.tight_layout(); plt.show()